In [0]:
# COMMAND ----------
from pyspark.sql import functions as F
from datetime import datetime

# Lecture Bronze
df = spark.table("pharma_catalog.bronze.bronze_suppliers")
display(df)

In [0]:
# COMMAND ----------
# Doublons
df_dedup = df.dropDuplicates(["supplier_id"])
doublons = df.count() - df_dedup.count()
print(f"🔁 Doublons détectés : {doublons}")

# COMMAND ----------
df_quality = df_dedup.withColumn(
    "rule_failed",
    F.when(F.col("supplier_id").isNull(), "null_supplier_id")
    .when(F.col("supplier_name").isNull(), "null_supplier_name")
    .when(F.col("country").isNull(), "null_country")
    .when(F.col("lead_time_days").isNull(), "null_lead_time_days")
    .when(F.col("lead_time_days") < 0, "lead_time_negatif")
    .when(F.col("contact_email").isNull(), "null_contact_email")
    .when(F.col("status").isNull(), "null_status")
    .otherwise(None)
)

passed = df_quality.filter(F.col("rule_failed").isNull()).drop("rule_failed")
failed = df_quality.filter(F.col("rule_failed").isNotNull())

print(f" Lignes OK : {passed.count()}")
print(f" Lignes rejetées : {failed.count()}")

In [0]:
# COMMAND ----------
if failed.count() > 0:
    df_quarantine = failed.withColumn("source_table", F.lit("bronze_suppliers")) \
                           .withColumn("processing_date", F.lit(datetime.now().strftime("%Y-%m-%d")))

    df_quarantine.write.mode("append").saveAsTable("pharma_catalog.silver_quarantine.quarantine")
    print(f" {failed.count()} ligne(s) envoyée(s) en quarantine")
else:
    print("Aucune ligne rejetée — quarantine vide")

In [0]:
# COMMAND ----------
# Envoi des lignes propres vers Silver
passed.write.mode("overwrite").saveAsTable("pharma_catalog.silver.silver_suppliers")

print(f"✅ {passed.count()} ligne(s) chargée(s) dans silver_suppliers")

In [0]:
# COMMAND ----------
display(spark.table("pharma_catalog.silver.silver_suppliers"))